# 01 — Data Preparation

Converts the raw Zenodo archives into tidy per-game event tables.

**Source:** Periáñez et al. (2017), *Churn prediction of mobile and online casual
games using play log data*, PLOS ONE 12(7).
Data: <https://zenodo.org/record/166035> (CC-BY 4.0) ·
Paper: <https://doi.org/10.1371/journal.pone.0180735>

Two defects in the published files are handled in code and documented below —
both are worth reporting in the methodology chapter.

In [1]:
import sys, warnings
from pathlib import Path

# Allow `from src import ...` when the notebook is run from notebooks/
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 60)
print("project root:", ROOT)

project root: D:\Player Churn\player-churn-ml


In [2]:
from src import config, etl

for f in sorted(config.RAW_DIR.glob("rawdata_game*")):
    print(f"{f.name:22s} {f.stat().st_size / 1e6:9.1f} MB")

rawdata_game1.csv            4.8 MB


rawdata_game2.zip          303.6 MB
rawdata_game3.zip          544.7 MB


## Defect 1 — Deflate64 compression

Seven of game2's eight archive parts are stored with compression **method 9
(Deflate64)**, which CPython's `zipfile` cannot open:

> `NotImplementedError: That compression method is not supported`

`src/zip64_reader.py` seeks to each member's local header and inflates the raw
stream incrementally, so no external 7-Zip install is needed and memory stays
flat regardless of member size (the parts expand to ~520 MB each).

In [3]:
import zipfile

with zipfile.ZipFile(config.RAW_DIR / "rawdata_game2.zip") as zf:
    info = pd.DataFrame([{
        "member": i.filename,
        "method": i.compress_type,
        "compressed_MB": round(i.compress_size / 1e6, 1),
        "uncompressed_MB": round(i.file_size / 1e6, 1),
    } for i in zf.infolist()])

print("method 8 = Deflate (supported), method 9 = Deflate64 (NOT supported by zipfile)")
print(f"total uncompressed: {info.uncompressed_MB.sum():,.0f} MB")
info

method 8 = Deflate (supported), method 9 = Deflate64 (NOT supported by zipfile)
total uncompressed: 4,296 MB


,member,method,compressed_MB,uncompressed_MB
0,part-00000,8,38.3,517.0
1,part-00001,9,36.5,523.5
2,part-00002,9,37.3,531.4
3,part-00003,9,37.1,526.8
4,part-00004,9,41.3,586.1
5,part-00005,9,37.1,530.1
6,part-00006,9,36.8,524.4
7,part-00007,9,39.2,557.0


## Defect 2 — Truncated JSON

`bulkplayers.json` in game3 is 933,730,335 bytes — exactly its declared size —
but the JSON is **truncated**: it ends with the final player list's `}]` and
never emits the outer object's closing brace. A strict streaming parser aborts
after ~411k of 713k records with `premature EOF`.

`src/etl.py` supplies the missing brace at end-of-stream. Only a closing brace
is added; **no record content is fabricated**, and the records lost to the
original truncation stay lost.

In [4]:
with zipfile.ZipFile(config.RAW_DIR / "rawdata_game3.zip") as zf:
    with zf.open("bulkplayers.json") as fh:
        total, tail = 0, b""
        while chunk := fh.read(1 << 24):
            total += len(chunk)
            tail = chunk[-120:]

print(f"bytes read: {total:,}")
print("final bytes:", tail.decode("utf8", "replace"))
print("\nends with a closing brace for the outer object?",
      tail.rstrip().endswith(b"}}"))

bytes read: 933,730,335
final bytes: },{"auth":true,"name":"Gryts","flair":5,"degree":120,"score":2,"points":0,"team":0,"events":"wBAFKQERYAEQZwERkwERHA=="}]

ends with a closing brace for the outer object? False


## Run the ETL

Skip this if `data/interim/*.parquet` already exist — game2 takes ~2 minutes.

In [5]:
REBUILD = False   # set True to regenerate the event tables

for game in config.GAMES:
    out = config.INTERIM_DIR / f"{game}_events.parquet"
    if REBUILD or not out.exists():
        etl.ETL[game]()
    else:
        print(f"{game}: using cached {out.name}")

game1: using cached game1_events.parquet
game2: using cached game2_events.parquet
game3: using cached game3_events.parquet


## Dataset summary

The event schema is identical across games so downstream code is game-agnostic;
columns a game cannot supply are simply null.

In [6]:
rows = []
for game in config.GAMES:
    df = pd.read_parquet(config.INTERIM_DIR / f"{game}_events.parquet")
    ts = pd.to_datetime(df["ts"], unit="s")
    rows.append({
        "game": game,
        "events": len(df),
        "players": df["player_id"].nunique(),
        "first": ts.min().date(),
        "last": ts.max().date(),
        "span_days": (ts.max() - ts.min()).days,
        "median_events_per_player": df.groupby("player_id").size().median(),
        "purchase_events": int((df["price"] > 0).sum()),
        "outcome_events": int(df["won"].notna().sum()),
        "authenticated": round(df["authed"].mean(), 3),
    })

summary = pd.DataFrame(rows)
summary.to_csv(config.TABLE_DIR / "dataset_summary.csv", index=False)
summary

,game,events,players,first,last,span_days,median_events_per_player,purchase_events,outcome_events,authenticated
0,game1,153929,25956,2015-01-13,2016-05-24,496,2.0,0,0,1.000
1,game2,7620167,79436,2014-06-15,2014-09-30,107,37.0,1750493,2436203,1.000
2,game3,3516753,91094,2015-05-25,2015-12-25,213,4.0,0,2743061,0.664


### Cross-check against the published figures

The source paper reports **10,874,958 log records** across **193,443 players**.
An independent extraction landing within a few percent is good evidence the ETL
is faithful.

In [7]:
print(f"extracted : {summary.events.sum():>12,} events | {summary.players.sum():>9,} players")
print(f"published : {10_874_958:>12,} events | {193_443:>9,} players")
print(f"difference: {summary.events.sum() / 10_874_958 - 1:>+11.1%} events | "
      f"{summary.players.sum() / 193_443 - 1:>+8.1%} players")

extracted :   11,290,849 events |   196,486 players
published :   10,874,958 events |   193,443 players
difference:       +3.8% events |    +1.6% players


### What each game contributes

| Game | Behavioural signal it uniquely provides |
|------|------------------------------------------|
| game1 | score progression only — the minimal-signal case |
| game2 | in-game purchases, race outcomes, `fail_state` (why an attempt failed), platform |
| game3 | match win/loss, match duration, registered-vs-guest identity |

In [8]:
df2 = pd.read_parquet(config.INTERIM_DIR / "game2_events.parquet")

print("game2 event types:")
display(df2["event"].value_counts().to_frame("count"))

print("\nrace outcomes (difficulty / frustration signal):")
display(df2["action"].value_counts().to_frame("count"))

print("\nwhy attempts failed:")
display(df2["fail_state"].value_counts().to_frame("count"))

print("\nplatform split:")
display(df2["platform"].value_counts().to_frame("count"))

game2 event types:


,count
event,
progress,4842981
softPurchase,1740979
init,472085
start,434413
install,80718
end,39476
softSale,9515



race outcomes (difficulty / frustration signal):


,count
action,
start,2406778
fail,982806
complete,735354
quit,718043



why attempts failed:


,count
fail_state,
character_name_gunnit,684923
Stuck,398456
Pit,343657
OutOfFuel,202474
PackageDestroyed,46231
Stationary,23981
Gunnit,545
on_launch,421
Complete,161



platform split:


,count
platform,
iOS,4305028
Android,3315139
